# Promoter Functional Classification — CNN + BiLSTM + GENERator
### No Leakage Version
- Label: `max(|expr|) > 3.0` across 6 conditions
- X_expr completely removed from model (source of label)
- No Sigmoid in model (BCEWithLogitsLoss handles it)
- AUC used for best model selection
- All-NaN rows removed before labeling

In [ ]:
# ── 1. Imports ───────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# ── 2. Load & Merge Data ─────────────────────────────────────────────────────
def clean_cols(df):
    df.columns = (
        df.columns.astype(str).str.strip().str.lower()
        .str.replace(r'[\s/]+', '_', regex=True)
        .str.replace(r'[,()\n]', '', regex=True)
    )
    return df

t1 = clean_cols(pd.read_excel('capstone_dataset.xlsx',
                               sheet_name='Supplementary Table 1', header=3))
t2 = clean_cols(pd.read_excel('capstone_dataset.xlsx',
                               sheet_name='Supplementary Table 2', header=3))

# Table 1 — sequences + metadata only
seq_col = next(c for c in t1.columns if 'sequence' in c)
t1 = t1[['gene', 'species', 'gc', 'strand', seq_col]].dropna()
t1 = t1.rename(columns={seq_col: 'sequence'})

# Table 2 — clean, remove all-NaN rows, then label
t2 = t2.rename(columns={'promoter': 'gene'}).dropna(subset=['gene'])
expr_cols = [c for c in t2.columns if c not in ('gene', 'species')]
t2[expr_cols] = t2[expr_cols].apply(pd.to_numeric, errors='coerce')

# Remove rows where ALL 6 expression values are NaN (pure missing data)
before = len(t2)
t2 = t2.dropna(subset=expr_cols, how='all')
print(f'Removed {before - len(t2)} all-NaN rows ({before} → {len(t2)})')

# Fill remaining NaN with 0 (not expressed in that condition)
t2[expr_cols] = t2[expr_cols].fillna(0)

# ── Threshold exploration ────────────────────────────────────────────────────
print('\n=== Label threshold exploration ===')
max_abs = t2[expr_cols].abs().max(axis=1)
print(max_abs.describe())
print()
for thresh in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5]:
    pos = (max_abs > thresh).sum()
    neg = (max_abs <= thresh).sum()
    print(f'  Threshold {thresh:.1f} → Functional: {pos:>6} | '
          f'Non-functional: {neg:>6} | Ratio: {pos/max(neg,1):.1f}:1')

# ── Set label threshold ───────────────────────────────────────────────────────
# 3.0 gives ~1.6:1 ratio — balanced and biologically meaningful
LABEL_THRESH = 3.0
t2['label'] = (t2[expr_cols].abs().max(axis=1) > LABEL_THRESH).astype(int)
print(f'\nUsing LABEL_THRESH = {LABEL_THRESH}')
print(t2['label'].value_counts())

# Merge — label ONLY, expression columns intentionally excluded from model
data = pd.merge(
    t1,
    t2[['gene', 'label']],   # ← NO expr_cols here
    on='gene', how='inner'
).reset_index(drop=True)

print('\nMerged shape:', data.shape)
print(f"Functional    (1): {data['label'].sum()}")
print(f"Non-functional(0): {(data['label']==0).sum()}")
print('data columns:', data.columns.tolist())

genes_all   = data['gene'].values
species_all = data['species'].values

In [ ]:
# ── 3. One-Hot Encode Sequences ───────────────────────────────────────────────
# X_expr intentionally NOT created — it is the source of the label (leakage)
MAX_LEN  = 200
BASE_MAP = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1]}

def one_hot_encode(seq, max_len=MAX_LEN):
    seq = str(seq).upper()[:max_len]
    arr = np.zeros((max_len, 4), dtype=np.float32)
    for i, b in enumerate(seq):
        if b in BASE_MAP:
            arr[i] = BASE_MAP[b]
    return arr

X_seq = np.array([one_hot_encode(s) for s in data['sequence']], dtype=np.float32)
y     = data['label'].values

print('X_seq:', X_seq.shape)
print(f'Functional: {y.sum()} | Non-functional: {(y==0).sum()}')

In [ ]:
# ── 4. Train / Val / Test Split ───────────────────────────────────────────────
idx = np.arange(len(y))
idx_train, idx_temp, y_train, y_temp = train_test_split(
    idx, y, test_size=0.30, stratify=y, random_state=42)
idx_val, idx_test, y_val, y_test = train_test_split(
    idx_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

X_seq_train, X_seq_val, X_seq_test = (
    X_seq[idx_train], X_seq[idx_val], X_seq[idx_test])

print(f'Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}')
print(f'Train → Functional: {y_train.sum()} | Non-functional: {(y_train==0).sum()}')
print(f'Val   → Functional: {y_val.sum()}   | Non-functional: {(y_val==0).sum()}')
print(f'Test  → Functional: {y_test.sum()}  | Non-functional: {(y_test==0).sum()}')

In [ ]:
# ── 5. GENERator Embeddings (run once, cached to disk) ───────────────────────
import os

GEN_CACHE = 'X_gen.npy'

if os.path.exists(GEN_CACHE):
    X_gen = np.load(GEN_CACHE)
    print('Loaded cached GENERator embeddings:', X_gen.shape)
    assert X_gen.shape[0] == len(data), \
        f'Cache mismatch! Cache has {X_gen.shape[0]} rows but data has {len(data)}. Delete X_gen.npy and rerun.'
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM

    tokenizer = AutoTokenizer.from_pretrained(
        'GenerTeam/GENERator-v2-eukaryote-1.2b-base', trust_remote_code=True)
    gen_model = AutoModelForCausalLM.from_pretrained(
        'GenerTeam/GENERator-v2-eukaryote-1.2b-base',
        torch_dtype=torch.float16          # fp16 for speed
    ).to(device).eval()

    def truncate_to_6(seq):
        r = len(seq) % 6
        return seq[r:] if r else seq

    @torch.no_grad()
    def generator_embeddings(seqs, batch_size=128):
        tokenizer.padding_side = 'left'
        out_all = []
        for i in range(0, len(seqs), batch_size):
            batch = [tokenizer.bos_token + truncate_to_6(s.upper())
                     for s in seqs[i:i+batch_size]]
            inputs  = tokenizer(batch, padding=True, return_tensors='pt').to(device)
            outputs = gen_model(**inputs, output_hidden_states=True)
            h    = outputs.hidden_states[-1].float()
            mask = inputs['attention_mask'].unsqueeze(-1).float()
            emb  = (h * mask).sum(1) / mask.sum(1)
            out_all.append(emb.cpu())
            if (i // batch_size) % 10 == 0:
                print(f'  Embedded {min(i+batch_size, len(seqs))}/{len(seqs)}')
        return torch.cat(out_all).numpy()

    X_gen = generator_embeddings(data['sequence'].values)
    np.save(GEN_CACHE, X_gen)
    print('Saved GENERator embeddings:', X_gen.shape)

In [ ]:
# ── 6. Dataset & DataLoaders ─────────────────────────────────────────────────
# PromoterDataset takes ONLY sequences + GENERator embeddings
# X_expr is intentionally excluded — it is the source of the label
class PromoterDataset(Dataset):
    def __init__(self, seqs, gens, labels=None):
        self.seqs   = torch.from_numpy(seqs.astype(np.float32))
        self.gens   = torch.from_numpy(gens.astype(np.float32))
        self.labels = (None if labels is None
                       else torch.tensor(labels, dtype=torch.float32).unsqueeze(1))

    def __len__(self): return len(self.seqs)

    def __getitem__(self, i):
        if self.labels is not None:
            return self.seqs[i], self.gens[i], self.labels[i]
        return self.seqs[i], self.gens[i]

BATCH = 128

train_ds = PromoterDataset(X_seq_train, X_gen[idx_train], y_train)
val_ds   = PromoterDataset(X_seq_val,   X_gen[idx_val],   y_val)
test_ds  = PromoterDataset(X_seq_test,  X_gen[idx_test],  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False,
                          num_workers=2, pin_memory=True)

# Sanity check — each batch should have 3 items: seq, gen, label
sample = next(iter(train_loader))
print('Items per batch (should be 3):', len(sample))
print('Shapes:', [x.shape for x in sample])
print('DataLoaders ready')

In [ ]:
# ── 7. Model ─────────────────────────────────────────────────────────────────
# No expr branch, no Sigmoid (BCEWithLogitsLoss handles sigmoid internally)
class PromoterHybridModel(nn.Module):
    def __init__(self, gen_dim, dropout=0.3):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=8),
            nn.ReLU(),
            nn.MaxPool1d(4)
        )
        self.bilstm = nn.LSTM(
            input_size=64, hidden_size=64,
            batch_first=True, bidirectional=True
        )
        self.seq_fc = nn.Linear(128, 64)
        self.gen_fc = nn.Linear(gen_dim, 64)
        self.drop   = nn.Dropout(dropout)

        # Raw logits — NO Sigmoid
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, seq, gen):
        x = self.cnn(seq.permute(0, 2, 1)).permute(0, 2, 1)
        _, (h, _) = self.bilstm(x)
        seq_feat = self.drop(self.seq_fc(torch.cat([h[-2], h[-1]], dim=1)))
        gen_feat = self.drop(self.gen_fc(gen))
        return self.classifier(torch.cat([seq_feat, gen_feat], dim=1))

model = PromoterHybridModel(gen_dim=X_gen.shape[1]).to(device)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── 8. Training ───────────────────────────────────────────────────────────────
# At 1.6:1 ratio, standard BCE is sufficient — no pos_weight or FocalLoss needed
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', patience=3, factor=0.5)

EPOCHS = 25
best_val_auc, best_state = 0.0, None
history = {'train_loss': [], 'val_loss': [], 'val_auc': []}

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, all_p, all_l = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for seq_b, gen_b, lab_b in loader:   # no expr_b
            seq_b  = seq_b.to(device)
            gen_b  = gen_b.to(device)
            lab_b  = lab_b.to(device)
            if train: optimizer.zero_grad()
            logits = model(seq_b, gen_b)      # no expr
            loss   = criterion(logits, lab_b)
            if train:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            all_p.append(torch.sigmoid(logits).detach().cpu().numpy().ravel())
            all_l.append(lab_b.cpu().numpy().ravel())
    probs  = np.concatenate(all_p)
    labels = np.concatenate(all_l).astype(int)
    try:
        auc = roc_auc_score(labels, probs)
    except ValueError:
        auc = 0.5
    return total_loss / len(loader), auc

print(f'\n{"Epoch":>5} | {"Train Loss":>10} {"Train AUC":>10} | {"Val Loss":>10} {"Val AUC":>10}')
print('-' * 65)

for ep in range(1, EPOCHS + 1):
    tr_loss, tr_auc = run_epoch(train_loader, train=True)
    vl_loss, vl_auc = run_epoch(val_loader,   train=False)
    scheduler.step(vl_auc)
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['val_auc'].append(vl_auc)
    marker = ' ✓' if vl_auc > best_val_auc else ''
    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
    print(f'{ep:>5} | {tr_loss:>10.4f} {tr_auc:>10.4f} | '
          f'{vl_loss:>10.4f} {vl_auc:>10.4f}{marker}')

model.load_state_dict(best_state)
print(f'\nBest Val AUC: {best_val_auc:.4f}')

In [ ]:
# ── 9. Training Curves ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
axes[1].plot(history['val_auc'], color='green', label='Val AUC')
axes[1].axhline(0.5, color='red', linestyle='--', label='Random baseline')
axes[1].set_title('Validation AUC'); axes[1].legend(); axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

In [ ]:
# ── 10. Optimal Threshold (from validation set) ───────────────────────────────
model.eval()
val_p, val_l = [], []
with torch.no_grad():
    for seq_b, gen_b, lab_b in val_loader:
        logits = model(seq_b.to(device), gen_b.to(device))
        val_p.append(torch.sigmoid(logits).cpu().numpy().ravel())
        val_l.append(lab_b.numpy().ravel())

val_probs  = np.concatenate(val_p)
val_labels = np.concatenate(val_l).astype(int)

best_thresh, best_f1 = 0.5, 0.0
for t in np.arange(0.05, 0.95, 0.01):
    f = f1_score(val_labels, (val_probs > t).astype(int), zero_division=0)
    if f > best_f1:
        best_f1, best_thresh = f, t

print(f'Optimal threshold : {best_thresh:.2f}')
print(f'Val F1 at threshold: {best_f1:.4f}')
print(f'Val prob range     : {val_probs.min():.4f} – {val_probs.max():.4f}')
print(f'Val AUC            : {roc_auc_score(val_labels, val_probs):.4f}')

In [ ]:
# ── 11. Test Set Evaluation ───────────────────────────────────────────────────
test_p, test_l = [], []
with torch.no_grad():
    for seq_b, gen_b, lab_b in test_loader:
        logits = model(seq_b.to(device), gen_b.to(device))
        test_p.append(torch.sigmoid(logits).cpu().numpy().ravel())
        test_l.append(lab_b.numpy().ravel())

probs  = np.concatenate(test_p)
y_true = np.concatenate(test_l).astype(int)
preds  = (probs > best_thresh).astype(int)

print(f'=== Test Results (threshold = {best_thresh:.2f}) ===')
print(f'Accuracy  : {accuracy_score(y_true, preds):.4f}   ← ignore at imbalance')
print(f'Precision : {precision_score(y_true, preds, zero_division=0):.4f}')
print(f'Recall    : {recall_score(y_true, preds, zero_division=0):.4f}')
print(f'F1        : {f1_score(y_true, preds, zero_division=0):.4f}')
print(f'ROC AUC   : {roc_auc_score(y_true, probs):.4f}   ← most important')
print()
print('Prediction distribution:')
unique, counts = np.unique(preds, return_counts=True)
for u, c in zip(unique, counts):
    label = 'Functional' if u == 1 else 'Non-functional'
    print(f'  {label}: {c}')

In [ ]:
# ── 12. Per-Species Confusion Matrix & ROC ────────────────────────────────────
output_df = pd.DataFrame({
    'Promoter'       : genes_all[idx_test],
    'Species'        : species_all[idx_test],
    'True_Label'     : ['Functional' if v == 1 else 'Non-functional' for v in y_true],
    'Predicted_Label': ['Functional' if v == 1 else 'Non-functional' for v in preds],
    'Probability'    : probs
})

species_list = output_df['Species'].unique()
fig, axes = plt.subplots(len(species_list), 2,
                          figsize=(12, 4 * len(species_list)))
if len(species_list) == 1:
    axes = axes[np.newaxis, :]

for row, sp in enumerate(species_list):
    sub  = output_df[output_df['Species'] == sp]
    y_t  = sub['True_Label'].map({'Non-functional': 0, 'Functional': 1})
    y_p  = sub['Predicted_Label'].map({'Non-functional': 0, 'Functional': 1})
    y_pr = sub['Probability']

    ConfusionMatrixDisplay(
        confusion_matrix(y_t, y_p),
        display_labels=['Non-func', 'Functional']
    ).plot(ax=axes[row, 0], cmap='Blues', colorbar=False)
    axes[row, 0].set_title(f'{sp} — Confusion Matrix')

    try:
        fpr, tpr, _ = roc_curve(y_t, y_pr)
        auc = roc_auc_score(y_t, y_pr)
        axes[row, 1].plot(fpr, tpr, label=f'AUC = {auc:.3f}')
    except ValueError:
        axes[row, 1].text(0.5, 0.5, 'Only one class in species',
                          ha='center', va='center')
    axes[row, 1].plot([0,1],[0,1],'--', color='grey')
    axes[row, 1].set_xlabel('FPR'); axes[row, 1].set_ylabel('TPR')
    axes[row, 1].set_title(f'{sp} — ROC Curve')
    axes[row, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── 13. Full Dataset Predictions ──────────────────────────────────────────────
full_ds     = PromoterDataset(X_seq, X_gen)   # no labels
full_loader = DataLoader(full_ds, batch_size=BATCH, shuffle=False,
                         num_workers=2, pin_memory=True)

model.eval()
all_full = []
with torch.no_grad():
    for seq_b, gen_b in full_loader:
        logits = model(seq_b.to(device), gen_b.to(device))
        all_full.append(torch.sigmoid(logits).cpu().numpy().ravel())

all_probs = np.concatenate(all_full)

plt.figure(figsize=(8, 4))
plt.hist(all_probs, bins=60, edgecolor='black')
plt.axvline(best_thresh, color='red', linestyle='--',
            label=f'Threshold = {best_thresh:.2f}')
plt.xlabel('Predicted Probability (Functional)')
plt.ylabel('Count')
plt.title('Predicted Probability Distribution — Full Dataset')
plt.legend()
plt.show()

print('Predicted Non-functional:', (all_probs < best_thresh).sum())
print('Predicted Functional    :', (all_probs >= best_thresh).sum())

In [ ]:
# ── 14. Save Results ──────────────────────────────────────────────────────────
result_df = pd.DataFrame({
    'Gene'       : genes_all,
    'Species'    : species_all,
    'Probability': all_probs,
    'Prediction' : ['Functional' if p >= best_thresh else 'Non-functional'
                    for p in all_probs]
})
result_df.to_csv('promoter_predictions.csv', index=False)
print('Saved → promoter_predictions.csv')
result_df.head(10)